I just copied over all of the code from the from the original notebook and I am beginning to split the data.

In [21]:
# Installing missing libraries
%pip install pandas numpy scikit-learn matplotlib seaborn


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [22]:
# Importing some libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [23]:
# Importing the Telco dataset and inspecting it
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [24]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [26]:
# Total charges is an object stored as a string
# I need to convert them to integers and ensure there are no missing values
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [27]:
df.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [28]:
df['TotalCharges'].dtype

dtype('float64')

In [29]:
df = df.dropna()

In [30]:
# Dropping customerID so the model doesn't try to learn patterns from it
# I will save it just in case we need it later
customer_ids = df['customerID'].copy()
df = df.drop('customerID', axis=1)

In [31]:
df['Contract'].unique()

<StringArray>
['Month-to-month', 'One year', 'Two year']
Length: 3, dtype: str

In [32]:
# THIS CELL CAN BE DELETED I WAS JUST FIGURING OUT ENCODING WITH IT

contract_encoded = pd.get_dummies(
    df[['Contract']],
    dtype=int
)

contract_encoded.head()

,Contract_Month-to-month,Contract_One year,Contract_Two year
0,1,0,0
1,0,1,0
2,1,0,0
3,0,1,0
4,1,0,0


In [33]:
categorical_cols = df.select_dtypes(include=['object', 'string']).columns

print(categorical_cols)

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'Churn'],
      dtype='str')


In [34]:
# Separate the features from the target variable
X = df.drop(columns=['Churn'])
y = df['Churn']

# Identify categorical features in X
categorical_cols = X.select_dtypes(include=['object', 'string']).columns

print(categorical_cols)

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='str')


In [35]:
# Convert the target variable to binary values
y = y.map({'No': 0, 'Yes': 1})

# Check the results
print(y.value_counts())

Churn
0    5163
1    1869
Name: count, dtype: int64


I am thinking 5-Fold Cross Validation with an 80-20 split. 80% of the data for training and 20% to test.

In [36]:
# Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=88,
    stratify=y
)

In [37]:
# Checking the dimensions of the training and test sets
print("X_train;", X_train.shape)
print("X_test;", X_test.shape)

print("y_train;", y_train.shape)
print("y_test;", y_test.shape)

X_train; (5625, 19)
X_test; (1407, 19)
y_train; (5625,)
y_test; (1407,)


There are 5,625 customers in the training set and 1,407 in the testing set. There are 19 X features because I removed customerID and I made Churn the target variable (y)

In [38]:
from sklearn.model_selection import StratifiedKFold

# Setting up 5-fold stratified cross-validation
# This is just the configuration
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=88
)

print(cv)

StratifiedKFold(n_splits=5, random_state=88, shuffle=True)


In [39]:
# Identifying categorical and numerical features within the training set
categorical_cols = X_train.select_dtypes(
    include=['object', 'string']
).columns

numerical_cols = X_train.select_dtypes(
    include=['number']
).columns

print("Categorical features:", categorical_cols)
print("Numerical features:", numerical_cols)

Categorical features: Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='str')
Numerical features: Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='str')


In [40]:
from sklearn.preprocessing import OneHotEncoder

# Creates an encoder for categorical features
encoder = OneHotEncoder(handle_unknown='ignore')

# Prints it so I know it is correct
print(encoder)

OneHotEncoder(handle_unknown='ignore')


In [43]:
from sklearn.compose import ColumnTransformer

# Defines how categorical and numerical features are to be processed
preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', encoder, categorical_cols),
        ('numerical', 'passthrough', numerical_cols)
    ]
)

print(preprocessor)

ColumnTransformer(transformers=[('categorical',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='str')),
                                ('numerical', 'passthrough',
                                 Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='str'))])
